# VerSe CT analysis — GPU Colab
Run cells from top to bottom. Select Runtime → Change runtime type → GPU first.

This standalone notebook first installs dependencies and downloads the VerSe19 validation dataset (~2.16 GB),
then verifies and loads the author-provided Task056_VerSe checkpoint (~242 MB). Choose `all` to preload all six
VerSe archives. Raw training datasets are not required to analyze uploaded scans; no training is performed.
The checkpoint is distributed by [CTSpine1K](https://github.com/MIRACLE-Center/CTSpine1K);
its task name is VerSe and it predicts C1–L6. No CTSpinoPelvic1K model is used.

Upload one or several de-identified CT volumes (.nii, .nii.gz, .nrrd), or one DICOM series per ZIP.
NIfTI/NRRD must contain CT intensities in HU. Oblique scans, irregular DICOM spacing and multiple series
are rejected with an explanation. Predictions return to the input geometry; model preprocessing uses its learned spacing.
Output ZIP: PDF and HTML reports, consistently colored tagged sagittal/coronal/axial PNGs,
per-vertebra views/masks, combined label map, CT, CSV/JSON measurements and logs.

Research segmentation only. Model numbering is provisional; this checkpoint has no T13 class and does not
diagnose fractures or classify congenital anomalies. Dataset: [official VerSe](https://github.com/anjany/verse).
Runtime: [nnUNet v1](https://github.com/MIC-DKFZ/nnUNet/tree/nnunetv1).
Re-running the run cell skips completed unchanged cases; start a new runtime for a fresh upload batch.


In [ ]:
DEVICE = 'gpu'
import os, sys, subprocess, json, shutil, time, hashlib, zipfile
from pathlib import Path
ROOT = Path('/content/verse_' + DEVICE)
ROOT.mkdir(parents=True, exist_ok=True)
PY = ROOT / 'venv/bin/python'
if not PY.exists():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(ROOT / 'venv')], check=True)
subprocess.run([str(PY), '-m', 'pip', 'install', '-q',
    'nnunet==1.7.1', 'numpy==1.26.4', 'scipy==1.13.1', 'scikit-image==0.24.0',
    'scikit-learn==1.5.2', 'pandas==2.2.3', 'matplotlib==3.9.4',
    'batchgenerators==0.25.1', 'nibabel==5.3.2', 'SimpleITK==2.4.1',
    'pydicom==3.0.1', 'gdown==5.2.0'], check=True)
ENV = os.environ.copy()
ENV.update(MPLCONFIGDIR=str(ROOT / 'mpl'), nnUNet_raw_data_base=str(ROOT / 'raw'),
           nnUNet_preprocessed=str(ROOT / 'preprocessed'), RESULTS_FOLDER=str(ROOT / 'models'),
           OMP_NUM_THREADS='2', nnUNet_def_n_proc='1', TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD='1')
# Legacy checkpoint loading is limited to the SHA256-verified author checkpoint below.
if DEVICE == 'cpu':
    ENV['CUDA_VISIBLE_DEVICES'] = ''
for key in ('nnUNet_raw_data_base', 'nnUNet_preprocessed', 'RESULTS_FOLDER', 'MPLCONFIGDIR'):
    Path(ENV[key]).mkdir(parents=True, exist_ok=True)
check = "import torch; print('PyTorch',torch.__version__); "
check += "assert torch.cuda.is_available(), 'Select a GPU runtime first'" if DEVICE == 'gpu' else "print('CPU runtime ready')"
subprocess.run([str(PY), '-c', check], env=ENV, check=True)
print('Dependencies ready. Separate environment avoids restarting the notebook kernel.')


## Load datasets and checkpoint

In [ ]:
# Preload reference data and all checkpoint assets BEFORE asking for your CT.
# 'validation' downloads the full ~2.16 GB VerSe19 validation archive.
# 'all' downloads all six VerSe19/20 archives and needs substantially more storage.
DATASET_SCOPE = 'validation' # @param ['validation', 'all']
DATA = ROOT / 'datasets'
DATA.mkdir(exist_ok=True)
MODEL = ROOT / 'models/3d_fullres/Task056_VerSe/nnUNetTrainerV2__nnUNetPlansv2.1'
(MODEL / 'all').mkdir(parents=True, exist_ok=True)
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8*1024**2), b''): h.update(block)
    return h.hexdigest()
assets = {
 'plans.pkl': ('1tjCxFW5H11ccio7Kqbywfo0U3i9SLKkD', '5dfbab18873e1e6c34e4edb28fda53e919ba4fcf19ca25085fee100e65332213'),
 'all/model_final_checkpoint.model.pkl': ('1FGYP61NbnEPx4ePEOx8sEPqEJSJI_TGL', '715ed0de4bbaa19fb51ea8ffbceb859da3b65b7f3e0df1c2dcf99d1d3872f4cf'),
 'all/model_final_checkpoint.model': ('1XT1LrGDrn_mctsGM6lb8-opSKPWrWqvj', 'b506b4283bf6b84f949d66b8f7bd3fd3a18000343c11568415cf90614bc7bec7')}
resources = {'model_source':'https://github.com/MIRACLE-Center/CTSpine1K',
             'dataset_source':'https://github.com/anjany/verse', 'model':{}, 'datasets':[]}
for name, (file_id, checksum) in assets.items():
    target = MODEL / name
    if not target.exists() or sha256(target) != checksum:
        part = target.with_suffix(target.suffix + '.part')
        url = 'https://drive.usercontent.google.com/download?id='+file_id+'&export=download&confirm=t'
        subprocess.run(['curl','-fL','--retry','5','--max-time','3600','-o',str(part),url], check=False)
        if not part.exists() or sha256(part) != checksum:
            subprocess.run([str(PY),'-m','gdown',file_id,'-O',str(part)], check=True)
        if sha256(part) != checksum:
            raise RuntimeError('Checkpoint checksum mismatch: '+name)
        part.replace(target)
    resources['model'][name] = checksum
splits = ['verse19validation'] if DATASET_SCOPE == 'validation' else [
    'verse19training','verse19validation','verse19test','verse20training','verse20validation','verse20test']
for split in splits:
    target = DATA / ('dataset-'+split+'.zip')
    marker = target.with_suffix('.verified.json')
    if not (target.exists() and marker.exists() and json.loads(marker.read_text())['bytes'] == target.stat().st_size):
        import urllib.request
        url = 'https://s3.bonescreen.de/public/VerSe-complete/'+target.name
        request = urllib.request.Request(url, method='HEAD')
        with urllib.request.urlopen(request, timeout=120) as response:
            size = int(response.headers.get('Content-Length', 0))
        if not size:
            raise RuntimeError('Could not establish dataset size. Rerun this cell later.')
        part = target.with_suffix('.zip.part')
        needed = max(0, size - (part.stat().st_size if part.exists() else 0))
        if shutil.disk_usage(ROOT).free < needed + 5*1024**3:
            raise RuntimeError('Insufficient disk for '+split+'. Select validation or use a larger runtime.')
        print('Downloading/resuming', split, flush=True)
        subprocess.run(['curl','-fL','--retry','5','-C','-','-o',str(part),url], check=True)
        if part.stat().st_size != size:
            raise RuntimeError('Dataset size mismatch')
        with zipfile.ZipFile(part) as z:
            if z.testzip(): raise RuntimeError('Dataset CRC validation failed')
            index = [{'name':i.filename,'bytes':i.file_size} for i in z.infolist()]
        part.replace(target)
        marker.write_text(json.dumps({'bytes':size,'sha256':sha256(target),'members':index}))
    resources['datasets'].append({'archive':target.name, **json.loads(marker.read_text())})
(ROOT / 'resources.json').write_text(json.dumps(resources, indent=2))
# Keep archives compressed to avoid duplicating gigabytes; contents are indexed and CRC checked.
subprocess.run([str(PY), '-c', 'import pickle,sys; p=pickle.load(open(sys.argv[1],"rb")); assert p["num_classes"]==25; print("Model plans verified; classes C1-L6")', str(MODEL/'plans.pkl')], env=ENV, check=True)
print('VerSe data and verified model ready. Training datasets are not needed for pretrained inference.')


## Load embedded analysis code

In [ ]:
(ROOT / "verse_pipeline.py").write_text('"""Embedded in both notebooks; standalone VerSe checkpoint inference and reporting."""\nimport os\nos.environ.setdefault(\'MPLCONFIGDIR\', \'/tmp/verse_matplotlib\')\nimport argparse, csv, hashlib, html, json, shutil, time, zipfile\nfrom pathlib import Path\nimport nibabel as nib\nimport numpy as np\nimport SimpleITK as sitk\nfrom scipy import ndimage\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.backends.backend_pdf import PdfPages\nfrom matplotlib.colors import ListedColormap, BoundaryNorm, to_hex\nfrom nnunet.preprocessing.preprocessing import get_do_separate_z, get_lowres_axis, resample_data_or_seg\n\nNAMES = {**{i: f\'C{i}\' for i in range(1, 8)},\n         **{i: f\'T{i-7}\' for i in range(8, 20)},\n         **{i: f\'L{i-19}\' for i in range(20, 26)}}\nCOLORS = [(0, 0, 0, 0)] + [tuple(plt.cm.hsv((i * .61803398875) % 1)) for i in range(1, 26)]\nCMAP = ListedColormap(COLORS)\nNORM = BoundaryNorm(np.arange(-.5, 26.5), 26)\nLIMITS = (\'Research segmentation report; not a diagnostic CT interpretation. \'\n          \'Labels require whole-spine anatomical counting. Missing labels may be outside the scan \'\n          \'or missed by the model. L6 is a model label, not confirmation of lumbarization. \'\n          \'This checkpoint cannot predict T13, sacrum or coccyx, or classify fractures, tumors or Castellvi types.\')\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(8 * 1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef safe_extract(source, target):\n    target = Path(target).resolve()\n    target.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(source) as z:\n        if sum(i.file_size for i in z.infolist()) > shutil.disk_usage(target).free - 2 * 1024**3:\n            raise RuntimeError(\'Insufficient disk space to extract ZIP.\')\n        for i in z.infolist():\n            p = (target / i.filename).resolve()\n            if not p.is_relative_to(target) or (i.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError(\'Unsafe ZIP member: \' + i.filename)\n        z.extractall(target)\n\ndef convert(source, out):\n    source, out = Path(source), Path(out)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    if source.suffix.lower() == \'.zip\':\n        root = out.parent / \'dicom_extracted\'\n        safe_extract(source, root)\n        import pydicom\n        groups = {}\n        for p in sorted(root.rglob(\'*\')):\n            if not p.is_file():\n                continue\n            try:\n                ds = pydicom.dcmread(p, stop_before_pixels=True)\n            except Exception:\n                continue\n            if getattr(ds, \'Modality\', \'\') != \'CT\' or not hasattr(ds, \'ImagePositionPatient\'):\n                continue\n            if int(getattr(ds, \'NumberOfFrames\', 1)) != 1:\n                raise ValueError(\'Enhanced multiframe DICOM: convert to NIfTI before upload.\')\n            groups.setdefault(str(ds.SeriesInstanceUID), []).append((p, ds))\n        if len(groups) != 1:\n            raise ValueError(f\'ZIP contains {len(groups)} CT series. Upload exactly one CT series per ZIP.\')\n        records = next(iter(groups.values()))\n        if len(records) < 3:\n            raise ValueError(\'At least three CT slices are required.\')\n        orient = np.array(records[0][1].ImageOrientationPatient, float)\n        normal = np.cross(orient[:3], orient[3:])\n        records.sort(key=lambda r: np.dot(np.array(r[1].ImagePositionPatient, float), normal))\n        positions = np.array([np.array(d.ImagePositionPatient, float) for _, d in records])\n        for _, d in records:\n            if not np.allclose(d.ImageOrientationPatient, orient, atol=1e-4):\n                raise ValueError(\'Inconsistent DICOM orientation.\')\n            if not np.allclose(d.PixelSpacing, records[0][1].PixelSpacing, atol=1e-4):\n                raise ValueError(\'Inconsistent pixel spacing.\')\n        gaps = np.diff(positions @ normal)\n        if np.any(gaps <= 0) or not np.allclose(gaps, np.median(gaps), rtol=.02, atol=.05):\n            raise ValueError(\'Duplicate, missing or irregularly spaced DICOM slices.\')\n        if not np.allclose(np.diff(positions, axis=0), gaps[:, None] * normal, atol=.1):\n            raise ValueError(\'Gantry tilt/shear: convert with a validated DICOM converter first.\')\n        reader = sitk.ImageSeriesReader()\n        reader.SetFileNames([str(p) for p, _ in records])\n        img = reader.Execute()\n    else:\n        if not str(source).lower().endswith((\'.nii\', \'.nii.gz\', \'.nrrd\')):\n            raise ValueError(\'Supported: NIfTI, self-contained NRRD, or ZIP of one CT DICOM series.\')\n        img = sitk.ReadImage(str(source))\n    if img.GetDimension() != 3 or img.GetNumberOfComponentsPerPixel() != 1:\n        raise ValueError(\'Expected scalar 3D CT.\')\n    arr = sitk.GetArrayViewFromImage(img)\n    if not np.isfinite(arr).all() or np.ptp(arr) == 0:\n        raise ValueError(\'CT is empty, constant or non-finite.\')\n    if min(img.GetSpacing()) <= 0:\n        raise ValueError(\'Invalid CT spacing.\')\n    # Remove metadata strings from the derived NIfTI; uploaded source stays untouched.\n    for key in img.GetMetaDataKeys():\n        img.EraseMetaData(key)\n    sitk.WriteImage(img, str(out), True)\n    return img\n\ndef infer(ct, prediction, model, work, device, tta):\n    import torch\n    from nnunet.training.network_training.nnUNetTrainerV2 import nnUNetTrainerV2\n    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))\n    if device == \'gpu\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Enable a GPU Colab runtime and rerun.\')\n    original = sitk.ReadImage(str(ct))\n    direction = np.array(original.GetDirection()).reshape(3, 3)\n    # Match the repository training orientation (LPS). Reject oblique input rather than\n    # silently applying the upstream rounding-based orientation code to it.\n    if not np.allclose(direction, np.round(direction), atol=1e-3):\n        raise ValueError(\'Oblique CT must first be resampled onto an orthogonal grid.\')\n    normalized = sitk.DICOMOrient(original, \'LPS\')\n    normalized_path = work / \'normalized_0000.nii.gz\'\n    sitk.WriteImage(normalized, str(normalized_path), True)\n    trainer = nnUNetTrainerV2(str(model / \'plans.pkl\'), \'all\',\n                            output_folder=str(work / \'trainer\'), dataset_directory=str(work),\n                            stage=1, fp16=(device == \'gpu\'))\n    trainer.initialize(training=False)\n    trainer.load_checkpoint(str(model / \'all/model_final_checkpoint.model\'), train=False)\n    trainer.network.to(\'cuda\' if device == \'gpu\' else \'cpu\')\n    trainer.network.eval()\n    trainer.data_aug_params[\'do_mirror\'] = bool(tta)\n    pred_lps = work / \'prediction_lps.nii.gz\'\n    print(\'preprocessing...\', flush=True)\n    data, _, properties = trainer.preprocess_patient([str(normalized_path)])\n    print(\'predicting...\', flush=True)\n    seg, _ = trainer.predict_preprocessed_data_return_seg_and_softmax(\n        data,\n        do_mirroring=trainer.data_aug_params[\'do_mirror\'],\n        mirror_axes=trainer.data_aug_params[\'mirror_axes\'],\n        use_sliding_window=True,\n        step_size=0.5,\n        use_gaussian=True,\n        pad_border_mode=\'constant\',\n        pad_kwargs={\'constant_values\': 0},\n        verbose=True,\n        all_in_gpu=False,\n        mixed_precision=device == \'gpu\')\n    del data\n    seg = seg.transpose(trainer.transpose_backward)\n    target_shape = properties[\'size_after_cropping\']\n    if tuple(seg.shape) != tuple(target_shape):\n        if get_do_separate_z(properties[\'original_spacing\']):\n            lowres_axis = get_lowres_axis(properties[\'original_spacing\'])\n        elif get_do_separate_z(properties[\'spacing_after_resampling\']):\n            lowres_axis = get_lowres_axis(properties[\'spacing_after_resampling\'])\n        else:\n            lowres_axis = None\n        if lowres_axis is not None and len(lowres_axis) != 1:\n            lowres_axis = None\n        print(\'low-memory label resample:\', seg.shape, \'->\', target_shape, flush=True)\n        seg = resample_data_or_seg(seg[None].astype(np.uint8), target_shape, is_seg=True,\n                                   axis=lowres_axis, order=0,\n                                   do_separate_z=lowres_axis is not None,\n                                   order_z=0)[0].astype(np.uint8)\n    original_shape = properties[\'original_size_of_raw_data\']\n    bbox = properties.get(\'crop_bbox\')\n    if bbox is not None:\n        restored_seg = np.zeros(original_shape, dtype=np.uint8)\n        for c in range(3):\n            bbox[c][1] = min(bbox[c][0] + seg.shape[c], original_shape[c])\n        restored_seg[bbox[0][0]:bbox[0][1],\n                     bbox[1][0]:bbox[1][1],\n                     bbox[2][0]:bbox[2][1]] = seg[:bbox[0][1]-bbox[0][0],\n                                                   :bbox[1][1]-bbox[1][0],\n                                                   :bbox[2][1]-bbox[2][0]]\n    else:\n        restored_seg = seg\n    seg_img = sitk.GetImageFromArray(restored_seg.astype(np.uint8))\n    seg_img.SetSpacing(properties[\'itk_spacing\'])\n    seg_img.SetOrigin(properties[\'itk_origin\'])\n    seg_img.SetDirection(properties[\'itk_direction\'])\n    sitk.WriteImage(seg_img, str(pred_lps), True)\n    restored = sitk.Resample(sitk.ReadImage(str(pred_lps)), original,\n                            sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)\n    sitk.WriteImage(restored, str(prediction), True)\n\ndef report(ct_path, pred_path, out, metadata):\n    out = Path(out)\n    for folder in (\'images\', \'masks\'):\n        (out / folder).mkdir(parents=True, exist_ok=True)\n    ct_img, pred_img = nib.load(str(ct_path)), nib.load(str(pred_path))\n    if ct_img.shape != pred_img.shape or not np.allclose(ct_img.affine, pred_img.affine, atol=1e-4):\n        raise ValueError(\'CT/mask geometry mismatch.\')\n    pred = np.asanyarray(pred_img.dataobj)\n    labels = np.unique(pred)\n    if not np.isfinite(pred).all() or not np.equal(pred, np.round(pred)).all() or labels.min() < 0 or labels.max() > 25:\n        raise ValueError(\'Invalid labels for this checkpoint.\')\n    present = [int(x) for x in labels if x > 0]\n    if not present:\n        raise ValueError(\'Empty segmentation; no successful report will be produced.\')\n    voxel_ml = abs(np.linalg.det(pred_img.affine[:3, :3])) / 1000\n    rows = []\n    for lab in present:\n        mask = pred == lab\n        count = int(mask.sum())\n        center = ndimage.center_of_mass(mask)\n        world = nib.affines.apply_affine(pred_img.affine, center)\n        touches = any(np.any(np.take(mask, i, axis=a)) for a in range(3) for i in (0, -1))\n        tags = [\'predicted_label\', \'verify_numbering\']\n        if touches:\n            tags.append(\'touches_scan_boundary\')\n        if lab == 25:\n            tags.append(\'L6_label_requires_anatomical_confirmation\')\n        rows.append(dict(label_id=lab, label=NAMES[lab], color=to_hex(COLORS[lab]),\n                         voxels=count, volume_ml=round(count * voxel_ml, 3),\n                         centroid_R_mm=round(float(world[0]), 3), centroid_A_mm=round(float(world[1]), 3),\n                         centroid_S_mm=round(float(world[2]), 3), tags=\'; \'.join(tags)))\n        header = pred_img.header.copy()\n        header.set_data_dtype(np.uint8)\n        nib.save(nib.Nifti1Image(mask.astype(np.uint8), pred_img.affine, header),\n                 out / \'masks\' / f\'{NAMES[lab]}.nii.gz\')\n    with (out / \'measurements.csv\').open(\'w\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    ct_c = nib.as_closest_canonical(ct_img)\n    pc = np.asanyarray(nib.as_closest_canonical(pred_img).dataobj)\n    cc = np.asanyarray(ct_c.dataobj)\n    spacing = ct_c.header.get_zooms()[:3]\n    summary = dict(metadata, shape=list(ct_img.shape), spacing_mm=list(map(float, ct_img.header.get_zooms()[:3])),\n                   affine=ct_img.affine.tolist(), labels=rows, geometry_verified=True,\n                   nonzero_voxels=int(np.count_nonzero(pred)), limitations=LIMITS,\n                   model=\'MIRACLE-Center/CTSpine1K Task056_VerSe; 25 foreground classes\',\n                   input_sha256=digest(ct_path), prediction_sha256=digest(pred_path))\n    (out / \'report.json\').write_text(json.dumps(summary, indent=2))\n    planes = [(0, \'sagittal\', \'P\', \'A\', \'S\', \'I\'),\n              (1, \'coronal\', \'L\', \'R\', \'S\', \'I\'),\n              (2, \'axial\', \'L\', \'R\', \'A\', \'P\')]\n    images = []\n    with PdfPages(out / \'CT_report.pdf\') as pdf:\n        fig = plt.figure(figsize=(8.27, 11.69))\n        fig.text(.06, .95, \'VerSe CT segmentation report\', fontsize=18, va=\'top\')\n        import textwrap\n        intro = f"Case: {metadata.get(\'case_id\', \'test\')} | Device: {metadata.get(\'device\', \'test\')}\\n"\n        intro += f"Shape: {ct_img.shape} | Spacing (mm): {tuple(round(float(s), 3) for s in ct_img.header.get_zooms()[:3])}\\n"\n        intro += \'Geometry verified. Bone display window: level 400 / width 1800 HU.\\n\\n\'\n        intro += \'\\n\'.join(textwrap.wrap(LIMITS, 88))\n        fig.text(.06, .89, intro, fontsize=10, va=\'top\')\n        for n, row in enumerate(rows):\n            y = .65 - n * .020\n            fig.text(.07, y, \'\\u25a0\', color=row[\'color\'], fontsize=12)\n            fig.text(.10, y, f"{row[\'label\']:4} {row[\'volume_ml\']:8.2f} mL  |  {row[\'voxels\']} voxels", fontsize=9)\n        pdf.savefig(fig)\n        plt.close(fig)\n        def render(axis, plane, index, name, edges):\n            image = np.take(cc, index, axis=axis).T\n            mask = np.take(pc, index, axis=axis).T\n            dims = [d for d in range(3) if d != axis]\n            extent = (0, image.shape[1]*spacing[dims[0]], 0, image.shape[0]*spacing[dims[1]])\n            fig, ax = plt.subplots(figsize=(8.27, 10), layout=\'constrained\')\n            ax.imshow(image, cmap=\'gray\', vmin=-500, vmax=1300, origin=\'lower\', extent=extent)\n            ax.imshow(np.ma.masked_equal(mask, 0), cmap=CMAP, norm=NORM, alpha=.48,\n                      origin=\'lower\', extent=extent, interpolation=\'nearest\')\n            for lab in np.unique(mask):\n                if lab <= 0:\n                    continue\n                yy, xx = ndimage.center_of_mass(mask == lab)\n                ax.text((xx+.5)*spacing[dims[0]], (yy+.5)*spacing[dims[1]], NAMES[int(lab)],\n                        color=\'white\', fontsize=9, ha=\'center\',\n                        bbox=dict(facecolor=COLORS[int(lab)], alpha=.85, edgecolor=\'white\', pad=2))\n            left, right, top, bottom = edges\n            for x, y, t in ((.01,.5,left),(.98,.5,right),(.5,.99,top),(.5,.01,bottom)):\n                ax.text(x,y,t,transform=ax.transAxes,color=\'cyan\',ha=\'center\',va=\'center\',weight=\'bold\')\n            ax.set_title(f\'{name} | {plane} slice {index}\\nPredicted labels — numbering requires review\', fontsize=11)\n            ax.set_xlabel(\'mm\'); ax.set_ylabel(\'mm\')\n            path = out / \'images\' / f\'{name}_{plane}.png\'\n            fig.savefig(path, dpi=150)\n            pdf.savefig(fig)\n            plt.close(fig)\n            images.append(path.relative_to(out).as_posix())\n        for axis, plane, *edges in planes:\n            occ = np.count_nonzero(pc, axis=tuple(a for a in range(3) if a != axis))\n            render(axis, plane, int(occ.argmax()), \'overview\', edges)\n        for lab in present:\n            for axis, plane, *edges in planes:\n                occ = np.count_nonzero(pc == lab, axis=tuple(a for a in range(3) if a != axis))\n                render(axis, plane, int(occ.argmax()), NAMES[lab], edges)\n    table = \'\'.join(f"<tr><td style=\'color:{r[\'color\']}\'>{r[\'label\']}</td><td>{r[\'volume_ml\']}</td><td>{html.escape(r[\'tags\'])}</td></tr>" for r in rows)\n    gallery = \'\'.join(f\'<figure><img loading="lazy" src="{p}"><figcaption>{p}</figcaption></figure>\' for p in images)\n    (out / \'CT_report.html\').write_text(\'<!doctype html><meta charset="utf-8"><title>VerSe CT report</title>\'\n        \'<style>body{font:16px sans-serif;max-width:1100px;margin:30px auto}td,th{padding:8px;border-bottom:1px solid #ccc}\'\n        \'img{max-width:100%}figure{display:inline-block;width:45%;vertical-align:top;margin:2%}</style>\'\n        f\'<h1>VerSe CT segmentation report</h1><p>{html.escape(LIMITS)}</p>\'\n        \'<p>Stable label colors; orientation letters and physical spacing shown on each image. \'\n        \'Each overview is one selected slice, not a projection of the entire spine.</p>\'\n        \'<table><tr><th>Predicted level</th><th>Volume mL</th><th>Tags</th></tr>\'+table+\'</table>\'+gallery)\n    return summary\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--config\', required=True)\n    args = p.parse_args()\n    cfg = json.loads(Path(args.config).read_text())\n    root = Path(cfg[\'root\'])\n    result = root / cfg[\'run_id\']\n    result.mkdir(exist_ok=True)\n    failures = []\n    for i, source in enumerate(cfg[\'inputs\'], 1):\n        case_id = f\'case_{i:03d}\'\n        out = result / case_id\n        out.mkdir(exist_ok=True)\n        signature = dict(source_sha256=digest(source), device=cfg[\'device\'], tta=cfg[\'tta\'])\n        marker = out / \'COMPLETE.json\'\n        if marker.exists() and json.loads(marker.read_text()) == signature:\n            print(\'Reusing completed:\', case_id, flush=True)\n            continue\n        work = root / (cfg[\'run_id\'] + \'_work\') / case_id\n        work.mkdir(parents=True, exist_ok=True)\n        try:\n            started = time.time()\n            print(\'Processing\', case_id, flush=True)\n            ct, pred = out / \'CT.nii.gz\', out / \'vertebrae.nii.gz\'\n            convert(source, ct)\n            infer(ct, pred, Path(cfg[\'model\']), work, cfg[\'device\'], cfg[\'tta\'])\n            report(ct, pred, out, dict(case_id=case_id, device=cfg[\'device\'], tta=cfg[\'tta\'],\n                                     elapsed_inference_seconds=round(time.time()-started, 2)))\n            marker.write_text(json.dumps(signature))\n        except Exception as e:\n            import traceback\n            traceback.print_exc()\n            failures.append(dict(case=case_id, error=str(e)))\n            (out / \'FAILED.txt\').write_text(str(e))\n    shutil.copy2(root / \'resources.json\', result / \'resources.json\')\n    (result / \'run_status.json\').write_text(json.dumps({\'failed\': failures, \'case_count\': len(cfg[\'inputs\'])}, indent=2))\n    manifest = {str(f.relative_to(result)): digest(f) for f in sorted(result.rglob(\'*\')) if f.is_file()}\n    (result / \'manifest_sha256.json\').write_text(json.dumps(manifest, indent=2))\n    archive = shutil.make_archive(str(result), \'zip\', result)\n    with zipfile.ZipFile(archive) as z:\n        if z.testzip():\n            raise RuntimeError(\'ZIP integrity check failed\')\n    print(\'ZIP:\', archive, flush=True)\n    if failures:\n        raise RuntimeError(f\'{len(failures)} case(s) failed. ZIP contains partial results and failure details.\')\n\nif __name__ == \'__main__\':\n    main()\n')
print("Analysis code ready")


## Upload CT files

In [ ]:
from google.colab import files
INPUT_DIR = ROOT / 'uploads'
INPUT_DIR.mkdir(exist_ok=True)
EXISTING_CT_PATHS = []  # Optional explicit paths already in /content or mounted Drive.
if EXISTING_CT_PATHS:
    INPUTS = [str(Path(p)) for p in EXISTING_CT_PATHS]
elif 'INPUTS' not in globals() or not all(Path(p).exists() for p in INPUTS):
    print('Upload CT .nii / .nii.gz / .nrrd, or ZIP containing exactly one DICOM CT series.')
    uploaded = files.upload()
    INPUTS = []
    for name, content in uploaded.items():
        target = INPUT_DIR / Path(name).name
        target.write_bytes(content)
        INPUTS.append(str(target))
    del uploaded
if not INPUTS or not all(Path(p).is_file() for p in INPUTS):
    raise ValueError('No valid CT files selected.')
print('Cases selected:', len(INPUTS))


## Run analysis

In [ ]:
# Same model, patch size and default settings in both notebooks.
# Mirroring is optional; enabling it increases inference time substantially.
TTA = False # @param {type:'boolean'}
batch_signature = [(str(p), sha256(p)) for p in INPUTS] + [(DEVICE, TTA)]
if 'RUN_ID' not in globals() or globals().get('LAST_BATCH') != batch_signature:
    RUN_ID = 'results_' + time.strftime('%Y%m%d_%H%M%S')
LAST_BATCH = batch_signature
CONFIG = dict(root=str(ROOT), model=str(MODEL), inputs=INPUTS, run_id=RUN_ID, device=DEVICE, tta=TTA)
(ROOT / 'run_config.json').write_text(json.dumps(CONFIG, indent=2))
log_path = ROOT / (RUN_ID + '.log')
print('Running full-resolution model. CPU may take hours; keep this runtime connected.', flush=True)
with log_path.open('w') as log:
    process = subprocess.Popen([str(PY), '-u', str(ROOT/'verse_pipeline.py'), '--config', str(ROOT/'run_config.json')],
                               env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            log.write(line); log.flush()
            print(line, end='')
        status = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
if status:
    print('One or more cases failed. Read the log above. The next cell can download any partial results.')
else:
    print('All cases completed. Run the final cell to preview and download.')


## Preview, save and download ZIP

In [ ]:
from IPython.display import display, Image, FileLink
archive = ROOT / (RUN_ID + '.zip')
if not archive.exists():
    raise RuntimeError('No ZIP was created. Check the inference log.')
with zipfile.ZipFile(archive, 'a', compression=zipfile.ZIP_DEFLATED) as z:
    if 'inference.log' not in z.namelist():
        z.write(log_path, 'inference.log')
with zipfile.ZipFile(archive) as z:
    if z.testzip(): raise RuntimeError('ZIP integrity check failed')
for path in sorted((ROOT/RUN_ID).glob('case_*/images/overview_sagittal.png')):
    display(Image(filename=str(path), width=650))
print('ZIP includes PDF/HTML reports, labeled PNGs in three planes, CT, masks, measurements, and logs.')
print('Run status:', (ROOT/RUN_ID/'run_status.json').read_text())
SAVE_TO_DRIVE = False # @param {type:'boolean'}
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/VerSe_results')
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / (DEVICE+'_'+archive.name)
    if target.exists():
        target = destination / (DEVICE+'_'+str(time.time_ns())+'_'+archive.name)
    shutil.copy2(archive, target)
    print('Drive copy:', target)
files.download(str(archive))
